In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

# Début de la mesure du temps global d'exécution
start_time_global = time.time()

# =============================================================================
# PARTIE 1 : CHARGEMENT DES DONNÉES & PRÉPARATION
# =============================================================================

path = r"C:\Users\pc\projet\return_nasdaq.m"
path_cov = r"C:\Users\pc\projet\covariance_nasdaq.m" 

try:
    data_returns = pd.read_csv(path, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    benefice = data_returns.to_numpy()
    print("--- Loading Returns Successful ---")
    print(f"Matrix Dimensions: {benefice.shape}")
except Exception as e:
    print(f"Error loading returns: {e}")
    benefice = np.random.randn(2196, 1)

try:
    data_cov = pd.read_csv(path_cov, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    covariance = data_cov.to_numpy()
    print("--- Loading Covariance Successful ---")
    print(f"Matrix Dimensions: {covariance.shape}")
except Exception as e:
    print(f"Error loading covariance: {e}")
    covariance = np.diag(np.random.uniform(0.01, 0.05, 2196))

# Configuration des paramètres de l'univers d'actifs
nb_asset = 2196
cardinal = 300

return_vect = benefice.flatten()
covar_matrix = covariance
variability = np.sqrt(np.diag(covar_matrix))

# Définition de la cible (Actifs avec rendement supérieur à la moyenne)
seuil = np.mean(return_vect)
target = (return_vect >= seuil).astype(int)

# Préparation des features pour le MLP
X = np.column_stack((return_vect, variability))
y = target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =============================================================================
# PARTIE 2 : CONFIGURATION ET ENTRAÎNEMENT DU MLP
# =============================================================================

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

print("\n--- Entraînement du MLP ---")
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

loss, accuracy, precision, recall = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision du MLP sur le test set : {accuracy:.2f}") 

# Prédiction sur l'ensemble des actifs
X_all_scaled = scaler.transform(X)
predictions_finales = (model.predict(X_all_scaled, verbose=0) > 0.5).astype(int).flatten()

# Sélection des indices retenus par le MLP
index_vector = [i for i in range(len(predictions_finales)) if predictions_finales[i] == 1]

# Filtrage par rendement pour ne garder que le top 'cardinal' (300)
return_index_vector = return_vect[index_vector]
indices_tries = sorted(range(len(return_index_vector)), key=lambda i: return_index_vector[i], reverse=True)
top_indices = indices_tries[:cardinal]
index_vector_vector = [index_vector[i] for i in top_indices]

# Création du masque binaire final
vecteur_portefeuille = np.zeros(nb_asset)
for i in index_vector_vector:
    vecteur_portefeuille[i] = 1

# --- CALCUL DES STATISTIQUES DEMANDÉES (AVEC & SANS BUDGET) ---
R_mlp = return_vect[index_vector_vector]
Cov_mlp = covar_matrix[np.ix_(index_vector_vector, index_vector_vector)]

# Indicateurs Bruts (Sans contrainte budgétaire, somme pure)
somme_rendement_brute = np.sum(R_mlp)
somme_risque_brute = np.sum(Cov_mlp)

# Indicateurs Étant Donné un Portefeuille Équipondéré (Avec allocation budgétaire de 1/300)
w_init = np.ones(cardinal) / cardinal
total_return_mlp_pondere = np.sum(w_init * R_mlp)
total_risk_mlp_pondere = w_init @ Cov_mlp @ w_init.T

print("\n" + "="*60)
print("             RÉSULTATS DU FILTRAGE MLP                      ")
print("="*60)
print(f"Nombre d'actifs sélectionnés par le MLP : {len(index_vector_vector)}")
print(f"SOMME BRUTE des rendements (sans budget): {somme_rendement_brute:.8f}")
print(f"SOMME BRUTE des risques (sans budget)   : {somme_risque_brute:.8f}")
print("-" * 60)
print(f"Rendement moyen attendu (équipondéré)   : {total_return_mlp_pondere:.8f}")
print(f"Risque total attendu (équipondéré)      : {total_risk_mlp_pondere:.8f}")
print("="*60 + "\n")

# =============================================================================
# PARTIE 3 : CODE NSGA-II OPTIMISÉ POUR L'EXPLORATION DES HAUTS RENDEMENTS
# =============================================================================
print("--- Lancement de l'optimisation NSGA-II (Mode Exploration) ---")

return_vect_filtered = return_vect.copy()
covar_matrix_filtered = covar_matrix.copy()

# Isolation stricte des 300 actifs choisis par le MLP
for i in range(nb_asset):
    if vecteur_portefeuille[i] == 0:
        return_vect_filtered[i] = 0
        covar_matrix_filtered[i, :] = 0
        covar_matrix_filtered[:, i] = 0

# Configuration de la population et des générations
nb_sol = 150                  
number_iteration_max = 200     
prob_crossover = 0.95         

# Opérateurs d'exploration agressive basés sur le comportement BLX-0.5
prob_mutation = 5 / cardinal  
mutation_strength = 0.25      

# Initialisation de la population initiale (Somme des poids = 1)
population = np.zeros((nb_sol, nb_asset))
for i in range(nb_sol):
    random_weights = np.random.uniform(0.1, 1.0, size=len(index_vector_vector))
    population[i, index_vector_vector] = random_weights / np.sum(random_weights)

def evaluate_population(pop):
    """Évaluation simultanée des deux objectifs (Rendement et Risque)"""
    returns = np.dot(pop, return_vect_filtered)
    risks = np.zeros(len(pop))
    for idx in range(len(pop)):
        w = pop[idx, :].reshape(1, -1)
        risks[idx] = w @ covar_matrix_filtered @ w.T
    return returns, risks

def fast_non_dominated_sort(returns, risks):
    """Tri non dominé efficace pour générer les fronts de Pareto"""
    pop_size = len(returns)
    domination_features = [[] for _ in range(pop_size)]
    domination_counters = np.zeros(pop_size, dtype=int)
    fronts = [[]]
    
    for p in range(pop_size):
        for q in range(pop_size):
            if (returns[p] >= returns[q] and risks[p] <= risks[q]) and (returns[p] > returns[q] or risks[p] < risks[q]):
                domination_features[p].append(q)
            elif (returns[q] >= returns[p] and risks[q] <= risks[p]) and (returns[q] > returns[p] or risks[q] < risks[p]):
                domination_counters[p] += 1
                
        if domination_counters[p] == 0:
            fronts[0].append(p)
            
    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in domination_features[p]:
                domination_counters[q] -= 1
                if domination_counters[q] == 0:
                    next_front.append(q)
        i += 1
        fronts.append(next_front)
    return fronts[:-1]

def calculate_crowding_distance(returns, risks, front):
    """Mesure de la densité locale pour préserver l'étalement des rendements"""
    distance = np.zeros(len(front))
    if len(front) <= 2:
        distance[:] = np.inf
        return distance
        
    obj_returns = returns[front]
    sort_idx_ret = np.argsort(obj_returns)
    distance[sort_idx_ret[0]] = np.inf
    distance[sort_idx_ret[-1]] = np.inf
    ret_range = np.max(obj_returns) - np.min(obj_returns)
    if ret_range > 0:
        for i in range(1, len(front) - 1):
            distance[sort_idx_ret[i]] += (obj_returns[sort_idx_ret[i+1]] - obj_returns[sort_idx_ret[i-1]]) / ret_range
            
    obj_risks = risks[front]
    sort_idx_risk = np.argsort(obj_risks)
    distance[sort_idx_risk[0]] = np.inf
    distance[sort_idx_risk[-1]] = np.inf
    risk_range = np.max(obj_risks) - np.min(obj_risks)
    if risk_range > 0:
        for i in range(1, len(front) - 1):
            distance[sort_idx_risk[i]] += (obj_risks[sort_idx_risk[i+1]] - obj_risks[sort_idx_risk[i-1]]) / risk_range
            
    return distance

# Boucle principale de l'algorithme génétique (500 générations)
for generation in range(1, number_iteration_max + 1):
    
    returns, risks = evaluate_population(population)
    offspring = np.zeros_like(population)
    
    indices = np.arange(nb_sol)
    np.random.shuffle(indices)
    
    for k in range(0, nb_sol, 2):
        parent1 = population[indices[k]]
        parent2 = population[indices[k+1]]
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        # --- CROISEMENT BLX-0.5 ---
        if np.random.rand() < prob_crossover:
            for asset_idx in index_vector_vector:
                pmin = min(parent1[asset_idx], parent2[asset_idx])
                pmax = max(parent1[asset_idx], parent2[asset_idx])
                diff = pmax - pmin
                
                child1[asset_idx] = np.random.uniform(pmin - 0.5 * diff, pmax + 0.5 * diff)
                child2[asset_idx] = np.random.uniform(pmin - 0.5 * diff, pmax + 0.5 * diff)
            
        # --- MUTATION GAUSSIENNE AMPLIFIÉE ---
        for asset_idx in index_vector_vector:
            if np.random.rand() < prob_mutation:
                child1[asset_idx] += np.random.normal(0, mutation_strength)
            if np.random.rand() < prob_mutation:
                child2[asset_idx] += np.random.normal(0, mutation_strength)
                
        # Assurer le maintien de la cardinalité (poids positifs minimaux pour le sous-univers)
        child1[child1 < 1e-6] = 1e-6
        child2[child2 < 1e-6] = 1e-6
        
        child1[vecteur_portefeuille == 0] = 0
        child2[vecteur_portefeuille == 0] = 0
        
        # Respect strict de la contrainte budgétaire globale (Somme des poids = 1)
        if np.sum(child1) > 0: child1 /= np.sum(child1)
        if np.sum(child2) > 0: child2 /= np.sum(child2)
        
        offspring[k] = child1
        offspring[k+1] = child2

    # Classement Élitisiste NSGA-II
    combined_pop = np.concatenate((population, offspring), axis=0)
    comb_returns, comb_risks = evaluate_population(combined_pop)
    
    fronts = fast_non_dominated_sort(comb_returns, comb_risks)
    first_front_indices = np.array(fronts[0])
    
    # Affichage structuré à chaque génération
    print(f"Iteration  {generation}")
    print(f"Non dominated selected solutions in iteration {generation} are: {first_front_indices}")
    print(f"Expected return of these solutions is: {comb_returns[first_front_indices]}")
    print(f"Risk of these solutions is: {comb_risks[first_front_indices]}\n")
    
    # Sélection de la population parente suivante pour l'itération à venir
    new_population = []
    front_idx = 0
    while len(new_population) + len(fronts[front_idx]) <= nb_sol:
        new_population.extend(fronts[front_idx])
        front_idx += 1
        if front_idx >= len(fronts): break
        
    if len(new_population) < nb_sol and front_idx < len(fronts):
        last_front = fronts[front_idx]
        distances = calculate_crowding_distance(comb_returns, comb_risks, last_front)
        sorted_last_front = [last_front[i] for i in np.argsort(distances)[::-1]]
        needed = nb_sol - len(new_population)
        new_population.extend(sorted_last_front[:needed])
        
    population = combined_pop[new_population]

# Calcul et affichage du temps final d'exécution globale
end_time_global = time.time()
execution_time = end_time_global - start_time_global

print("="*60)
print(f"Temps d'exécution total du script : {execution_time:.2f} secondes")
print("="*60)

--- Loading Returns Successful ---
Matrix Dimensions: (2196, 1)
--- Loading Covariance Successful ---
Matrix Dimensions: (2196, 2196)

--- Entraînement du MLP ---
Précision du MLP sur le test set : 1.00

             RÉSULTATS DU FILTRAGE MLP                      
Nombre d'actifs sélectionnés par le MLP : 300
SOMME BRUTE des rendements (sans budget): 3.26328264
SOMME BRUTE des risques (sans budget)   : 55.47454857
------------------------------------------------------------
Rendement moyen attendu (équipondéré)   : 0.01087761
Risque total attendu (équipondéré)      : 0.00061638

--- Lancement de l'optimisation NSGA-II (Mode Exploration) ---
Iteration  1
Non dominated selected solutions in iteration 1 are: [109 151 153 177 179 198 213 229 234 236 248 253]
Expected return of these solutions is: [0.01104044 0.01111087 0.01101861 0.01482966 0.01250471 0.01218083
 0.01139956 0.01200839 0.01282552 0.01126468 0.01215705 0.01165818]
Risk of these solutions is: [0.00060448 0.00061379 0.00057121